# Zadania na ocenę 01 - Od autograd do uczenia sieci i optymalizatorów

**Cel:** odtworzyć znane elementy: graf obliczeń, model `nn.Module`, funkcję straty, pętlę uczenia, SGD/Momentum/Adam i interpretację trajektorii na powierzchni straty.

**Jak pracować?** W komórkach oznaczonych `TODO` uzupełnij `...` albo miejsca opisane komentarzem. Uruchamiaj komórki po kolei. Komórki `assert` są testami, więc mogą zgłaszać błąd, dopóki nie wykonasz zadania.

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print("PyTorch:", torch.__version__)
print("Uruchamiamy na CPU.")

## 1. Autograd

Dla $q=x+y$ i $f=qz$ oraz $x=-2$, $y=5$, $z=-4$ oblicz ręcznie: $f$, $\partial f/\partial x$, $\partial f/\partial y$, $\partial f/\partial z$. Następnie **uzupełnij PyTorch**. 

In [ ]:
# TODO 1: zbuduj trzy tensory-liście wymagające gradientu.
x = ...
y = ...
z = ...

# TODO 2: forward oraz backward.
q = ...
f = ...
# ...

print("f =", f)
print("gradienty:", x.grad, y.grad, z.grad)

In [ ]:
# Test 1: włącz po wykonaniu TODO powyżej.
assert torch.is_tensor(f) and torch.isclose(f, torch.tensor(-12.0))
assert x.grad is not None and y.grad is not None and z.grad is not None
assert all(
    torch.isclose(v.grad, torch.tensor(expected))
    for v, expected in [(x, -4.0), (y, -4.0), (z, 3.0)]
)
print("Autograd: wynik i gradienty zgodne z regułą łańcuchową")

**Pytanie:** dlaczego dla `q = x+y` gradient względem `x` i `y` jest taki sam? Dlaczego `x.grad` jest wypełnione dopiero po `backward()`?

## 2. Własna sieć

Z danych 2D będziemy przewidywać jedną z dwóch klas. Przygotowano nieliniowy problem dwóch przeplatających się półokręgów. Dane i podział są gotowe, czas poświęcamy na budowę modelu.

In [ ]:
# Dane: dwie półobręcze + szum.
def make_two_moons(n=600, noise=0.12, seed=11):
    rng = np.random.default_rng(seed)
    n0 = n // 2
    t0 = rng.uniform(0, np.pi, n0)
    t1 = rng.uniform(0, np.pi, n - n0)
    a = np.column_stack([np.cos(t0), np.sin(t0)])
    c = np.column_stack([1 - np.cos(t1), 0.5 - np.sin(t1)])
    X = np.vstack([a, c]) + rng.normal(0, noise, size=(n, 2))
    y = np.r_[np.zeros(n0, dtype=np.int64), np.ones(n - n0, dtype=np.int64)]
    p = rng.permutation(n)
    return torch.tensor(X[p], dtype=torch.float32), torch.tensor(y[p], dtype=torch.long)


X_all, y_all = make_two_moons()
X_train, X_test = X_all[:480], X_all[480:]
y_train, y_test = y_all[:480], y_all[480:]
plt.figure(figsize=(6, 3.5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=10, alpha=0.7)
plt.title("Dane treningowe: czy model liniowy wystarczy?")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
# TODO 3: uzupełnij znany schemat Linear -> Tanh -> Linear.
class TinyMLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            # TODO: dodaj trzy moduły, zachowaj 2 wejścia i 2 logity na wyjściu.
        )

    def forward(self, x):
        # TODO: zwróć wynik sieci.
        return ...


model = TinyMLP(hidden=16)
logits = model(X_train[:8])
print("Wymiary:", X_train[:8].shape, logits.shape)
print("Liczba parametrów:", sum(p.numel() for p in model.parameters()))

In [ ]:
assert logits.shape == (8, 2), "Sieć powinna zwrócić 2 logity dla każdej z 8 próbek."
assert sum(p.numel() for p in model.parameters()) == (2 * 16 + 16 + 16 * 2 + 2)
print("✓ Poprawne wymiary i liczba parametrów")

**Pytanie:** co zmieni się po usunięciu `Tanh`? Czy dwie warstwy `Linear` bez nieliniowości wystarczą dla takich danych?

**Odpowiedź:** 

## 3. Funkcja straty

Uzupełnij kompletny pojedynczy krok treningu, a następnie wykorzystamy go w pętli. **Jako funkcję straty uzyj entropię krzyzową**.

In [ ]:
# TODO 4: wykonaj dokładnie JEDEN krok uczenia.
torch.manual_seed(7)
probe = TinyMLP(hidden=16)
opt_probe = torch.optim.SGD(probe.parameters(), lr=0.1)
weights_before = [p.detach().clone() for p in probe.parameters()]

# opt_probe.???
# pred = ???
# loss = ???
# loss.???
# opt_probe.???

# Po wykonaniu TODO odkomentuj poniższe linie, aby sprawdzić wynik.:
# print('Strata:', loss.item())
# print('Parametry zmienione:', any(not torch.equal(a, p) for a,p in zip(weights_before, probe.parameters())))

In [ ]:
# Test 3: odkomentuj po wykonaniu TODO 4.
# assert torch.is_tensor(loss) and loss.ndim == 0
# assert any(not torch.equal(a, p) for a,p in zip(weights_before, probe.parameters()))
# print('ok')

## 4. Porównanie optimizerów

Trzy modele muszą mieć **ten sam początek, dane, architekturę i liczbę epok**. Różnią się tylko metodą aktualizacji (a dla porównania praktycznej skuteczności dopuszczamy dobranie learning rate do metody).

In [ ]:
# Parametry porównania — można eksperymentować po zakończeniu zadania.
configs = {
    "SGD": (torch.optim.SGD, {"lr": 0.15}),
    "Momentum": (torch.optim.SGD, {"lr": 0.15, "momentum": 0.9}),
    "Adam": (torch.optim.Adam, {"lr": 0.03}),
}
EPOCHS = 10


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()


# TODO 5: uzupełnij trening i ewaluację dla każdej konfiguracji.
torch.manual_seed(123)
initial = TinyMLP(hidden=16)
start_state = copy.deepcopy(initial.state_dict())
history = {}

for name, (opt_class, kwargs) in configs.items():
    net = TinyMLP(hidden=16)
    net.load_state_dict(start_state)
    optimizer = opt_class(net.parameters(), **kwargs)
    losses, scores = [], []
    for epoch in range(EPOCHS):
        net.train()
        # TODO: zero_grad -> forward -> loss -> backward -> step
        # ...
        # TODO: zapisz bieżący train loss i test accuracy
        # ...
    history[name] = (losses, scores)

# Po zakończeniu TODO wykresy uruchom poniżej.

In [ ]:
# Wizualizacja gotowa — uwaga na ocenę testową: służy WYŁĄCZNIE do pomiaru.
assert all(
    len(v[0]) == EPOCHS and len(v[1]) == EPOCHS for v in history.values()
), "Dokończ TODO 5: po 100 wyników dla metody."
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
for name, (losses, scores) in history.items():
    ax[0].plot(losses, label=name)
    ax[1].plot(scores, label=name)
ax[0].set(title="Strata na treningu", xlabel="Epoka", ylabel="Cross-entropy")
ax[1].set(title="Accuracy na teście", xlabel="Epoka", ylabel="Accuracy", ylim=(0, 1))
for a in ax:
    a.grid(alpha=0.25)
    a.legend()
plt.show()
for name, (losses, scores) in history.items():
    print(f"{name:9s} loss={losses[-1]:.3f}  test acc={scores[-1]:.1%}")

**Kolejna część zadania:** spróbuj dobrać liczbę epok tak, aby na wykresach krzywych uczących było widać "plateau"

**Pytanie:** Czy najszybszy spadek train loss oznacza automatycznie najwyższe accuracy test? Dlaczego użycie identycznej wartości `lr` dla różnych rodzin optymalizatorów nie zawsze jest uczciwym porównaniem?

**Odpowiedź:**

## 5. Wykres funkcji straty

W rzeczywistej sieci strata zależy od wielu wag. Tutaj wybieramy **dwuwymiarową, wydłużoną "misę"** $L(w)=\tfrac12(w_1^2+20w_2^2)$, aby móc narysować poziomice i trajektorie.

In [ ]:
# TODO 6: implementacja funkcji i jednej trajektorii.
def bowl(w):
    # TODO: zwróć L(w)
    return ...


def trajectory(optimizer_class, kwargs, steps=50, start=(5.0, 4.0)):
    w = torch.tensor(start, dtype=torch.float32, requires_grad=True)
    optimizer = optimizer_class([w], **kwargs)
    points = [w.detach().clone().numpy()]
    for _ in range(steps):
        # TODO: zero_grad -> loss -> backward -> step
        # ...
        points.append(w.detach().clone().numpy())
    return np.array(points)


assert torch.isclose(bowl(torch.tensor([1.0, 2.0])), torch.tensor(40.5))
paths = {
    "SGD": trajectory(torch.optim.SGD, {"lr": 0.055}),
    "Momentum": trajectory(torch.optim.SGD, {"lr": 0.055, "momentum": 0.8}),
    "Adam": trajectory(torch.optim.Adam, {"lr": 0.35}),
}
assert all(
    not np.allclose(p[0], p[-1]) for p in paths.values()
), "Uzupełnij krok aktualizacji."

In [ ]:
u = np.linspace(-6, 6, 160)
v = np.linspace(-5, 5, 160)
U, V = np.meshgrid(u, v)
Z = 0.5 * (U**2 + 20 * V**2)
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
ax.contour(U, V, Z, levels=np.geomspace(0.5, 250, 17), colors="gray", linewidths=0.7)
for name, p in paths.items():
    ax.plot(p[:, 0], p[:, 1], marker=".", markersize=3, label=name)
ax.scatter([0], [0], marker="*", s=100, color="black", label="Minimum")
ax.set(
    xlabel="w₁",
    ylabel="w₂",
    title="Trajektorie na tej samej powierzchni straty",
    xlim=(-6, 6),
    ylim=(-5, 5),
)
ax.legend()
plt.show()

## 6. Wnioski

Zapisz **krótkie odpowiedzi**, wykorzystując własne wyniki:

1. Jak `backward()` łączy się z krokiem optymalizatora? Co zapisuje się w `param.grad`?
2. Po co potrzebna jest nieliniowość w naszej sieci?
3. Który optymalizator najszybciej zmniejszał loss, a który osiągnął najwyższe accuracy? Czy wynik musi być taki sam dla innych seedów?


**Odpowiedzi:**

Ad 1.

Ad 2.

Ad 3.